# Module 318 — GoodWords Spam Filter Baseline

This notebook implements the **Spam Filter Implementation** section only: load the UCI SMS messages, make word-count features, train Multinomial Naive Bayes, and evaluate held-out predictions. Later GoodWords sections can reuse the in-memory `vectorizer` and `classifier`. No attack logic is included here. Use the `AI` Python environment with pandas, NumPy, and scikit-learn installed.

The classifier sees numeric word counts, not message meaning as a person would understand it.

## Formula the code implements

Bayes' rule is `P(C|D) = P(D|C) P(C) / P(D)`. Read `P(C|D)` aloud as **“probability of class C given document D.”** `C` is spam or ham; `P(C)` is its prior; `P(D|C)` measures how likely the message is under that class. The model selects `arg max`—**“the argument that maximizes”**—the class score.

Naive Bayes approximates `P(D|C)` as the product, from `i=1` to `n`, of `P(w_i|C)`. Say `w_i` as **“w sub i.”** This assumes word features are conditionally independent given the class. Taking logs turns the product into the sum `log P(C) + sum_i log P(w_i|C)`, avoiding multiplication of many tiny probabilities.

With vocabulary `[free, hello]`, the text `free free hello` becomes the count vector `[2, 1]`.

In [ ]:
from io import BytesIO
from pathlib import Path
import csv
import html
import random
import re
import unicodedata
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

In [ ]:
# Fixed seeds make the random train/test split reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_PATH = DATA_DIR / 'sms_spam.csv'
print('Dataset cache:', DATASET_PATH.resolve())

## Load and cache the dataset

The first run downloads the course-linked UCI archive and writes a parsed CSV into `data/`. Reruns use that local cache. The archive is parsed in memory. Size and member-name checks catch an unexpected response before it is treated as training data.

In [ ]:
if DATASET_PATH.is_file():
    df = pd.read_csv(DATASET_PATH)
else:
    dataset_url = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
                   '00228/smsspamcollection.zip')
    with urlopen(dataset_url, timeout=30) as response:
        archive_bytes = response.read()
    if len(archive_bytes) > 5_000_000:
        raise ValueError('Unexpectedly large dataset archive')
    with ZipFile(BytesIO(archive_bytes)) as archive:
        member = archive.getinfo('SMSSpamCollection')
        if member.file_size > 2_000_000:
            raise ValueError('Unexpectedly large SMS file')
        with archive.open(member) as source:
            df = pd.read_csv(source, sep='\t', header=None,
                names=['label', 'message'], quoting=csv.QUOTE_NONE,
                encoding='utf-8')
    df.to_csv(DATASET_PATH, index=False)

assert list(df.columns) == ['label', 'message']
assert df['label'].isin(['ham', 'spam']).all()
print('Rows:', len(df))
print(df['label'].value_counts())
display(df.groupby('label', group_keys=False).head(2)[['label', 'message']])

## Normalize text but preserve useful clues

We normalize encoding, case, and whitespace while retaining numbers, currency, and selected punctuation. Aggressive cleaning can erase spam evidence before the classifier can learn it. A retained character only becomes a model feature if the vectorizer's token rules accept it.

In [ ]:
def minimal_clean(text):
    """Normalize HTML entities, Unicode, and whitespace."""
    text = html.unescape(text)
    text = unicodedata.normalize('NFKC', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_text(text):
    """Lowercase and retain common spam indicators."""
    text = text.lower()
    text = re.sub(r"[^\w\s£$€¥!?.,;:'\"-]", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['clean_message'] = df['message'].map(minimal_clean).map(clean_text)
before = len(df)
df = df.drop_duplicates(subset=['label', 'clean_message']).copy()
df = df[df['clean_message'].str.len() > 0].copy()
print('Rows removed:', before - len(df))
print('Rows retained:', len(df))
print('Class proportions (%):')
print((100 * df['label'].value_counts(normalize=True)).round(1))

## Split before fitting the vocabulary

The training data teaches both the vocabulary and class word statistics. The held-out test data measures performance after training. Fitting a vectorizer on all rows would expose vocabulary information from the test set. `stratify=y` keeps the class proportions similar in both groups.

In [ ]:
X_text = df['clean_message']
y = df['label']
train_text, test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, random_state=SEED, stratify=y
)
print('Training rows:', len(train_text), 'Test rows:', len(test_text))
print('Train class mix:\n', y_train.value_counts(normalize=True).round(3))
print('Test class mix:\n', y_test.value_counts(normalize=True).round(3))

## Create sparse word-count features

`fit_transform` learns token-to-column mappings from training messages and encodes their counts. `transform` uses that same mapping on test messages. A sparse matrix stores only nonzero entries instead of allocating memory for every absent word in every message.

In [ ]:
vectorizer = CountVectorizer(
    max_features=3000,
    token_pattern=r'\b\w+\b|[£$€¥]+|\d+|!!+|\?\?+|\.\.+',
    lowercase=True, stop_words='english'
)
X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)
feature_names = vectorizer.get_feature_names_out()
print('Train matrix (messages, features):', X_train.shape)
print('Test matrix (messages, features):', X_test.shape)
print('Storage format:', X_train.format)
print('Example vocabulary:', feature_names[:20].tolist())

## Train and evaluate Multinomial Naive Bayes

`fit` estimates class priors and smoothed word-count likelihoods. Accuracy is the share of all predictions that are correct. For spam, precision asks what fraction of messages predicted as spam truly are spam; recall asks what fraction of all actual spam messages the model catches. Because ham is the majority class, accuracy alone can hide poor spam detection.

In [ ]:
classifier = MultinomialNB()
classifier.fit(X_train, y_train)
test_predictions = classifier.predict(X_test)
print('Learned classes:', classifier.classes_.tolist())
print('Test accuracy:', round(accuracy_score(y_test, test_predictions), 4))
print('Confusion matrix (actual rows; predicted columns; ham, spam):')
print(confusion_matrix(y_test, test_predictions, labels=['ham', 'spam']))
print(classification_report(y_test, test_predictions,
    labels=['ham', 'spam'], target_names=['ham', 'spam'], zero_division=0))

## Baseline boundary and COAE recall

HTB's displayed accuracy and per-class metrics are run-specific examples; dataset, split, tokenization, and library versions can change them. This notebook learns the vocabulary from training rows only and avoids loading serialized Python pickle files. It stores the fitted `vectorizer` and `classifier` in memory for later sections.

**Remember:** Multinomial Naive Bayes adds word evidence by class; the bag-of-words representation ignores context; stratify the split; evaluate minority-class precision and recall, not accuracy alone. At this point we have built the target classifier only—no evasion code has been added.